In [25]:
from xml.etree import ElementTree

annotations = ElementTree.parse("/home/daniel/Downloads/annotations.xml").getroot()
image_info = annotations.findall("image")

In [28]:
from pathlib import Path

SPLIT = "test"

# Create the dataset structure.
root_dir = Path("plot_status_dataset") / SPLIT
root_dir.mkdir(exist_ok=True, parents=True)
headland_dir = root_dir / "headland"
headland_dir.mkdir(exist_ok=True)
in_plot_dir = root_dir / "in_plot"
in_plot_dir.mkdir(exist_ok=True)
between_plots_dir = root_dir / "between_plots"
between_plots_dir.mkdir(exist_ok=True)
    

In [29]:
import shutil

IMAGE_DIR = Path("/home/daniel/Downloads/boll_data") / SPLIT

def _copy_image(image: str, class_dir: Path) -> None:
    # Copies an image into the directory for a class.
    source_path = IMAGE_DIR / image
    if not source_path.exists():
        return
    shutil.copyfile(source_path, class_dir / image)

for image_node in image_info:
    image_name = image_node.get("name")
    tag = image_node.find("tag")
    if tag is None:
        # No tags means its in the headland.
        _copy_image(image_name, headland_dir)
    elif tag.find("attribute").text == "in_plot":
        # It is inside a plot.
        _copy_image(image_name, in_plot_dir)
    elif tag.find("attribute").text == "between_plots":
        # It is between plots.
        _copy_image(image_name, between_plots_dir)
    else:
        raise ValueError(f"Unexpected image attribute value {tag.find('attribute').text}")
        

In [23]:
import random

image_files = list(IMAGE_DIR.parent.iterdir())
small_dataset = random.sample(image_files, 70)
random.shuffle(small_dataset)
train_data = small_dataset[:50]
test_data = small_dataset[50:]

In [24]:
train_dir = IMAGE_DIR.parent / "train"
test_dir = IMAGE_DIR.parent / "test"
train_dir.mkdir(exist_ok=True)
test_dir.mkdir(exist_ok=True)

for image in train_data:
    shutil.copyfile(image, train_dir / image.name)
for image in test_data:
    shutil.copyfile(image, test_dir / image.name)